# SAGE sensitivity

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys

# Replace 'my_script_folder' with the actual path to your folder in Google Drive
script_path = '/content/drive/MyDrive/SAGE/Github'
sys.path.insert(0, script_path)

In [ ]:
from sage_sensitivity import *

Streaming output truncated to the last 5000 lines.
   PASS: Efficiency = BOL value at deg_eff = 0
   PASS: Efficiency = EOL value at deg_eff = 1
   PASS: Efficiency decreases monotonically
   PASS: Efficiency follows linear interpolation

[4] Cell Temperature
----------------------------------------
   PASS: T_cell = T_ambient when idle (no offset)
   PASS: T_cell = T_ambient + T_static when idle
   PASS: Discharge increases cell temperature
   PASS: Temperature rise matches heat generation model
   PASS: Lower efficiency produces more heating

[5] Calendar Aging Increment
----------------------------------------
   PASS: Higher temperature accelerates calendar aging
   PASS: Higher SOC accelerates calendar aging
   PASS: Calendar increment at T_ref matches SOC factor

[6] Cycle Aging Increment (battery-side power, cap_now)
----------------------------------------
   PASS: No cycle aging when idle
   PASS: Higher power increases cycle aging
   PASS: Cycle aging scales linearly with pow

## Original notebook follows

In [ ]:
# """
# SAGE Sensitivity Analysis - CSV-Configured, No Unnecessary Files
# """

# import pandas as pd
# import numpy as np
# from pathlib import Path
# import shutil
# import nbformat
# from time import time
# import gc

# SAGE_DIR = Path("/content/drive/MyDrive/SAGE")
# SAGE_NOTEBOOK = SAGE_DIR / "SAGE.ipynb"
# OUTPUT_DIR = Path("/content/drive/MyDrive/SAGE/sensitivity_results_final")
# SWEEPS_CSV = SAGE_DIR / "sensitivity_sweeps.csv"
# OUTPUT_DIR.mkdir(exist_ok=True)

# # Create example CSV if it doesn't exist
# if not SWEEPS_CSV.exists():
#     example_df = pd.DataFrame([
#         {'parameter': 'sigma_manufacturing_quality', 'values': '0.01, 0.02, 0.03, 0.05'},
#         {'parameter': 'vertical_temp_gradient_C', 'values': '2, 5, 8'},
#         {'parameter': 'beta_cal', 'values': '0.5, 0.6, 0.75'},
#         {'parameter': 'E_a_cal', 'values': '45000, 53000, 60000'},
#         {'parameter': 'E_a_cyc', 'values': '25000, 35000, 45000'},
#         {'parameter': 'container_baseline', 'values': '18, 22, 26, 30'},
#         {'parameter': 'num_discharge_hours', 'values': '2, 4, 6'},
#         {'parameter': 'SoH_EOL', 'values': '0.6, 0.7, 0.8'},
#     ])
#     example_df.to_csv(SWEEPS_CSV, index=False)
#     print(f"Created example sweep file: {SWEEPS_CSV}")
#     print("Edit this file to change parameters, then re-run.")

# # Load sweeps from CSV
# sweeps_df = pd.read_csv(SWEEPS_CSV)
# SWEEPS = {}
# for _, row in sweeps_df.iterrows():
#     param = row['parameter']
#     values = [float(v.strip()) for v in row['values'].split(',')]
#     SWEEPS[param] = values

# print(f"Loaded {len(SWEEPS)} parameters from {SWEEPS_CSV.name}:")
# for p, v in SWEEPS.items():
#     print(f"  {p}: {v}")

# # Check what's already done
# completed = set()
# for f in OUTPUT_DIR.glob("sage_fleet_lifespans_*.csv"):
#     run_name = f.stem.replace("sage_fleet_lifespans_", "")
#     completed.add(run_name)
# print(f"\nAlready completed: {len(completed)} runs")

# SENSITIVITY_FLEET_SIZE = 100

# with open(SAGE_NOTEBOOK, 'r', encoding='utf-8') as f:
#     nb = nbformat.read(f, as_version=4)

# sage_code_cells = [cell.source for cell in nb.cells if cell.cell_type == 'code']
# sage_code_cells = [c for c in sage_code_cells if '%matplotlib' not in c]

# def make_run_name(param_name, value):
#     val_str = f"{value:.4f}".replace(".", "p").rstrip("0").rstrip("p") or "0"
#     return f"{param_name}_{val_str}"

# def collect_outputs(run_name):
#     src = SAGE_DIR / "sage_fleet_lifespans.csv"
#     if src.exists():
#         dst = OUTPUT_DIR / f"{src.stem}_{run_name}{src.suffix}"
#         shutil.copy(src, dst)
#         return 1
#     return 0

# total_runs = sum(len(v) for v in SWEEPS.values())
# print("=" * 60)
# print(f"SAGE SENSITIVITY ANALYSIS - {total_runs} total, {total_runs - len(completed)} remaining")
# print(f"Output: {OUTPUT_DIR}")
# print("=" * 60)

# run_log = []
# run_count = 0
# t_start_total = time()

# for param_name, values in SWEEPS.items():
#     for value in values:
#         run_count += 1
#         run_name = make_run_name(param_name, value)

#         if run_name in completed:
#             print(f"[{run_count}/{total_runs}] {run_name} - SKIPPED (already done)")
#             continue

#         print(f"\n[{run_count}/{total_runs}] {run_name}")

#         t_start = time()
#         try:
#             run_globals = {'__name__': '__main__'}

#             for i, cell_code in enumerate(sage_code_cells):
#                 exec(cell_code, run_globals)
#                 if i == 1:
#                     run_globals['params_dict'][param_name] = value
#                     run_globals['params_dict']['fleet_size'] = SENSITIVITY_FLEET_SIZE

#             collect_outputs(run_name)
#             elapsed = (time() - t_start) / 60

#             df = pd.read_csv(SAGE_DIR / "sage_fleet_lifespans.csv")
#             mean_life = df['lifespan_years'].mean()

#             print(f"   {param_name}={value} → {mean_life:.1f} years ({elapsed:.1f} min)")
#             run_log.append({
#                 'parameter': param_name,
#                 'value': value,
#                 'mean_lifespan': mean_life,
#                 'status': 'success',
#             })

#         except Exception as e:
#             print(f"   ✗ Failed: {str(e)[:80]}")
#             run_log.append({
#                 'parameter': param_name,
#                 'value': value,
#                 'mean_lifespan': None,
#                 'status': 'failed',
#             })

#         del run_globals
#         gc.collect()

# log_df = pd.DataFrame(run_log)
# log_df.to_csv(OUTPUT_DIR / "sensitivity_run_log.csv", index=False)

# print(f"\n{'=' * 60}")
# print(f"COMPLETE - Total time: {(time() - t_start_total) / 60:.1f} min")
# print(f"Results: {OUTPUT_DIR}")
# print("=" * 60)

Streaming output truncated to the last 5000 lines.
[3] Efficiency Fade
----------------------------------------
   PASS: Efficiency = BOL value at deg_eff = 0
   PASS: Efficiency = EOL value at deg_eff = 1
   PASS: Efficiency decreases monotonically
   PASS: Efficiency follows linear interpolation

[4] Cell Temperature
----------------------------------------
   PASS: T_cell = T_ambient when idle (no offset)
   PASS: T_cell = T_ambient + T_static when idle
   PASS: Discharge increases cell temperature
   PASS: Temperature rise matches heat generation model
   PASS: Lower efficiency produces more heating

[5] Calendar Aging Increment
----------------------------------------
   PASS: Higher temperature accelerates calendar aging
   PASS: Higher SOC accelerates calendar aging
   PASS: Calendar increment at T_ref matches SOC factor

[6] Cycle Aging Increment (battery-side power, cap_now)
----------------------------------------
   PASS: No cycle aging when idle
   PASS: Higher power increa